# Aquaplanet

SPEEDY T31L8 over a slab ocean with thermodynamic sea ice. This run is one command:

```bash
python -m jem.main +configuration=aquaplanet-slab
```

The notebook below composes the same configuration in Python and calls `jem.runners.run(cfg)` -- the entry point `python -m jem.main` itself uses -- so it can plot what the run wrote.

In [ ]:
from pathlib import Path

from hydra import compose, initialize_config_module

import jem.config  # noqa: F401  -- registers the ${jem_data:}/${jcm_data:} resolvers
from jem import plot, runners

output_dir = (Path("output") / "01-01_aquaplanet").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## Run it

In [ ]:
with initialize_config_module(config_module="jem.config", version_base="1.3"):
    cfg = compose(config_name="config", overrides=[
        "+configuration=aquaplanet-slab",
        f"coupled_run.output_dir={output_dir}",
        "coupled_run.subsample=3",       # 10 records out of 30 coupled days
        "coupled_run.checkpoint_path=null",
    ])
result = runners.run(cfg)
result.steps_completed, [p.name for p in result.paths]

## What it wrote

One file per component per chunk, named after the coupled step its chunk starts at (`<component>-<first step>.nc`).

In [ ]:
atm = plot.open_output(output_dir, "atm")
ocn = plot.open_output(output_dir, "ocn")
seaice = plot.open_output(output_dir, "seaice")
list(ocn.data_vars)

## Plot

All three coupled components show up below: surface specific humidity (animated -- the field this example's top-level README entry is named after) and sea surface temperature from the atmosphere and ocean, and ice thickness from the sea-ice component.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# `level` is a sigma coordinate, surface-first, so selecting by its
# value (rather than `.isel(level=0)`) says so without relying on that
# ordering -- see "Output conventions" in
# docs/source/design/output.md.
humidity = atm["specific_humidity"].sel(level=1.0, method="nearest")
plot.map_plot(humidity.isel(time=-1), ax=axes[0],
              title="Surface specific humidity [kg/kg]")

sst = ocn["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst, ax=axes[1], title="Sea surface temperature [°C]")

plot.map_plot(seaice["ice_thickness"].isel(time=-1), ax=axes[2],
              title="Sea ice thickness [m]")
plt.tight_layout()

fig, ax = plt.subplots()
plot.area_mean(ocn["sea_surface_temperature"]).plot(ax=ax)
ax.set_ylabel("Area-mean SST [K]")

# Animate the same field the static panel above shows, and the field
# the top-level README's gif is named after.
ani = plot.animate_map(humidity, title="Surface specific humidity [kg/kg]")
ani.save(output_dir / "specific_humidity.gif", writer="pillow")